# Tugas Mandiri Teknik Kompilasi
## Lexer, Parser (EBNF), AST, dan TAC

Nama : Muhammad Listanto  
NIM : 231011400117  
Kelas : 06TPLE003  
Mata Kuliah : Teknik Kompilasi

In [4]:
import re


# ==========================================
# 1. DEFINISI AST
# ==========================================

class AST:
    pass


class BinOp(AST):
    def __init__(self, left, op, right):
        self.left = left
        self.op = op
        self.right = right


class Num(AST):
    def __init__(self, value):
        self.value = value


class Var(AST):
    def __init__(self, name):
        self.name = name


class ParserError(Exception):
    pass


# ==========================================
# 2. IMPLEMENTASI MINI COMPILER
# ==========================================

class MiniCompiler:

    def __init__(self, source, env):

        # Regex sudah ditambahkan operator ^
        self._tokens = iter(
            re.findall(
                r'[a-zA-Z_]\w*|\d+(?:\.\d+)?|[\^+*/()\-]',
                source
            ) + ['?']
        )

        self._current = None
        self._env = env
        self._temp_count = 0

        self.advance()

    def advance(self):
        try:
            self._current = next(self._tokens)

        except StopIteration:
            self._current = None

    def expect(self, expected):

        if self._current != expected and not (
            expected == "ID" and self._current.isalnum()
        ):

            raise ParserError(
                f"Expected {expected}, found {self._current}"
            )

        token = self._current
        self.advance()

        return token

    def factor(self):

        token = self._current

        # Number
        if token is not None and token.replace('.', '', 1).isdigit():

            self.advance()

            return Num(
                float(token) if '.' in token else int(token)
            )

        # Variable
        elif token and token.isalpha():

            # Analisis Semantik
            if token not in self._env:

                raise ParserError(
                    f"Semantic Error: Undefined variable '{token}'"
                )

            self.advance()

            return Var(token)

        # Parentheses
        elif token == '(':

            self.advance()

            node = self.expr()

            self.expect(')')

            return node

        raise ParserError(f"Unexpected token: {token}")

    # ==========================================
    # Fungsi power() untuk operator ^
    # ==========================================

    def power(self):

        node = self.factor()

        while self._current == '^':

            op = self._current

            self.advance()

            node = BinOp(
                left=node,
                op=op,
                right=self.factor()
            )

        return node

    # ==========================================
    # term() sekarang memanggil power()
    # ==========================================

    def term(self):

        node = self.power()

        while self._current in ('*', '/'):

            op = self._current

            self.advance()

            node = BinOp(
                left=node,
                op=op,
                right=self.power()
            )

        return node

    def expr(self):

        node = self.term()

        while self._current in ('+', '-'):

            op = self._current

            self.advance()

            node = BinOp(
                left=node,
                op=op,
                right=self.term()
            )

        return node

    # ==========================================
    # Generate TAC
    # ==========================================

    def generate_tac(self, node):

        if isinstance(node, Num):
            return str(node.value)

        if isinstance(node, Var):
            return node.name

        left_val = self.generate_tac(node.left)

        right_val = self.generate_tac(node.right)

        self._temp_count += 1

        temp_name = f"t{self._temp_count}"

        print(
            f"{temp_name} = {left_val} {node.op} {right_val}"
        )

        return temp_name

In [5]:
# ==========================================
# 3. UJI COBA
# ==========================================

source_code = "a ^ 2 + b * c"

symbol_table = {
    'a': 5,
    'b': 10,
    'c': 2
}

try:

    print("===================================")
    print("Mini Compiler")
    print("===================================")

    print(f"Input Source Code : {source_code}")

    compiler = MiniCompiler(
        source_code,
        symbol_table
    )

    ast_root = compiler.expr()

    print("\n--- Output Three Address Code (TAC) ---")

    compiler.generate_tac(ast_root)

except Exception as e:

    print(f"Error: {e}")

Mini Compiler
Input Source Code : a ^ 2 + b * c

--- Output Three Address Code (TAC) ---
t1 = a ^ 2
t2 = b * c
t3 = t1 + t2


In [6]:
# ==========================================
# 3. UJI COBA
# ==========================================

source_code = "a ^ 2 + b * c"

symbol_table = {
    'a': 5,
    'b': 10,
    'c': 2
}

try:

    print("===================================")
    print("Mini Compiler")
    print("===================================")

    print(f"Input Source Code : {source_code}")

    compiler = MiniCompiler(
        source_code,
        symbol_table
    )

    ast_root = compiler.expr()

    print("\n--- Output Three Address Code (TAC) ---")

    compiler.generate_tac(ast_root)

except Exception as e:

    print(f"Error: {e}")

Mini Compiler
Input Source Code : a ^ 2 + b * c

--- Output Three Address Code (TAC) ---
t1 = a ^ 2
t2 = b * c
t3 = t1 + t2


# 4. Jawaban Pertanyaan Refleksi

## 1. Mengapa fungsi power() harus dipanggil di dalam term(), bukan sebaliknya?

Karena operator pangkat (^) memiliki prioritas lebih tinggi dibanding operator perkalian (*) dan pembagian (/).

Dalam parsing recursive descent:

- expr() → menangani + dan -
- term() → menangani * dan /
- power() → menangani ^
- factor() → angka, variabel, dan kurung

Dengan hierarki tersebut, operasi pangkat akan diproses terlebih dahulu sebelum perkalian atau pembagian sesuai aturan Operator Precedence.

---

## 2. Apa yang terjadi pada fase Analisis Semantik jika variabel z digunakan tetapi tidak ada di symbol_table?

Fase Analisis Semantik akan mendeteksi error karena variabel tersebut belum didefinisikan.

Pengecekan dilakukan pada fungsi factor():

```python
if token not in self._env:

    

## 3. Mengapa instruksi untuk a ^ 2 harus muncul sebelum instruksi untuk + dalam TAC?

Karena TAC mengikuti urutan evaluasi berdasarkan prioritas operator dan dependency data.

Operasi pangkat harus dihitung terlebih dahulu agar hasilnya dapat digunakan pada operasi berikutnya.

Contoh urutan TAC:

t1 = a ^ 2
t2 = b * c
t3 = t1 + t2

Nilai t1 dan t2 harus tersedia terlebih dahulu sebelum proses penjumlahan dilakukan.